[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-3/lab-3.2-flash-attention-blind.ipynb)

# LAB·3.2 · Flash attention, blind build

**Hardware:** correctness anywhere; the 1.3x-of-reference gate needs a TPU runtime.

Rules of the blind build: your LAB·3.1 derivation and the Pallas you know are the only inputs. No reading Splash, no reading Tokamax, until `check` passes. Then diff yours against theirs and catalog every difference as algorithm, schedule, or feature; the differences are the syllabus for the rest of the stage.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
# one query block per grid step; the kv axis streams inside the kernel
def flash_kernel(q_ref, k_ref, v_ref, o_ref, *, block_kv):
    q = q_ref[...].astype(jnp.float32)
    n_kv = k_ref.shape[0]

    def step(j, state):
        m, l, acc = state
        kb = jax.lax.dynamic_slice_in_dim(k_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        vb = jax.lax.dynamic_slice_in_dim(v_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        s = q @ kb.T
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        l_new = l * alpha + jnp.sum(p, axis=-1)
        acc_new = acc * alpha[:, None] + p @ vb
        return m_new, l_new, acc_new

    m0 = jnp.full((q.shape[0],), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((q.shape[0],), jnp.float32)
    acc0 = jnp.zeros((q.shape[0], v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_kv // block_kv, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

In [ ]:
import functools

def flash(q, k, v, block_q=128, block_kv=128):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(flash_kernel, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[
            pl.BlockSpec((block_q, d), lambda i: (i, 0)),
            pl.BlockSpec(k.shape, lambda i: (0, 0)),
            pl.BlockSpec(v.shape, lambda i: (0, 0)),
        ],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        interpret=INTERP,
    )(q, k, v)

q = jax.random.normal(jax.random.key(0), (512, 64), jnp.float32)
k = jax.random.normal(jax.random.key(1), (1024, 64), jnp.float32)
v = jax.random.normal(jax.random.key(2), (1024, 64), jnp.float32)
ref = jax.nn.softmax(q @ k.T, axis=-1) @ v
check("flash forward", flash(q, k, v), ref, tol=1e-4)

## What the score matrix never did

Nothing in the kernel ever holds more than (block_q × block_kv) of S. The LAB·2.2 spill is gone by construction, not by fusion. Note the schedule caveat honestly: this version keeps whole K and V in VMEM per grid step, which caps sequence length; the production kernels block the KV axis in the grid too. That upgrade is your diff against Splash.

## Measure (TPU runtime)

Bench yours against `jax.nn.dot_product_attention` and, if available in your environment, a reference flash implementation, across seq 4k / 8k / 16k. Gate: within 1.3x at seq 8192. Record chip, dtype, shapes with every number and paste the blob into `bench/`.